# RT channel (8B) — Approach A: RT as binned tokens

We proved scale doesn't help (70B == 8B). The only way to break the thin-residual
ceiling is to give the model the **reaction-time** channel it never had.

**Key (your) insight:** Minitaur was trained choice-only, so it cannot predict RT.
We must FIRST train a population model that learns RT (**M_pop_RT**); only then can
the individual signal ride on the RT residual.

**Approach A** (this notebook): render a per-trial **log-RT decile token** inside
`<<>>` (e.g. `You press <<larger_later>>. <<rt7>>`). Because it's inside `<<>>` it is
loss-bearing, so the *entire existing pipeline* (masking / M_pop / surprise) works
unchanged — the model now predicts choice **and** RT.

Pipeline: **train M_pop_RT → surprise transfer on RT data → compare within-id to the
choice-only baseline (~0.14).** If it jumps, RT carries the missing individual signal.

## Get the RT data onto Drive (one-time)

RT-rendered transcripts for **all 33 tasks** (choice + log-RT decile per trial),
complete + retest. The RT-ceiling tasks (stroop/simon/ANT/stop-signal/…) — excluded
from the choice-only study because their individual signal lives in RT — are now
included, and are exactly where RT should help most.

**Upload the single zip `output_nl_rt.zip` (~17 MB)** to
`/content/drive/MyDrive/sro_minitaur/`. The setup cell unzips it automatically.

In [ ]:
# setup
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer
# liger-kernel = fused cross-entropy -> removes the [B,L,V] logits bottleneck -> bigger batch
!pip -q install peft bitsandbytes scikit-learn liger-kernel

# unzip the RT data (uploaded as output_nl_rt.zip)
RT = '/content/drive/MyDrive/sro_minitaur/output_nl_rt'
if not os.path.isdir(RT + '/complete'):
    !cd /content/drive/MyDrive/sro_minitaur && unzip -oq output_nl_rt.zip
assert os.path.isdir(RT + '/complete'), 'upload output_nl_rt.zip to /content/drive/MyDrive/sro_minitaur/ first'
print('RT tasks (complete):', len(os.listdir(RT + '/complete')))

In [ ]:
# 1) train M_pop_RT on ALL 33 tasks (choice + RT in the loss).
#    A100-40G + Liger: batch 8 fits (~29 GB); batch 12 OOMs (swiglu activations).
#    Look for "[liger] fused kernels enabled". If you still OOM -> --batch-size 6.
#    All 33 tasks ~3x data. (The Map/tokenize step is cached, so re-runs are fast.)
!cd /content/sro-minitaur-transfer && python scripts/finetune_mpop.py \
    --subset all \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --out /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --max-seq-len 4096 --batch-size 8 --grad-accum 2

In [ ]:
# 2a) surprise transfer on the SAME 11 tasks as the choice-only study (apples-to-apples).
#     Read the printed "within-domain mean top1" -> compare to choice-only 0.139 (raw) / 0.153 (M_pop).
!cd /content/sro-minitaur-transfer && python scripts/build_surprise_matrix.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --subset starting_subset --max-seq-len 4096 --batch-tokens 49152 --tag rt

In [ ]:
# 3) reference: choice-only baselines (starting_subset). Compare to the RT runs printed above.
import json
r = '/content/drive/MyDrive/sro_minitaur/results'
for tag, lab in [('8b_raw', 'choice-only raw 8B  (11 tasks)'), ('', 'choice-only M_pop   (11 tasks)')]:
    suf = f'_{tag}' if tag else ''
    try:
        s = json.load(open(f'{r}/surprise_matrix_summary{suf}.json'))
        print(f"{lab:32s} within={s['within']:.3f}  across={s['across']:.3f}")
    except FileNotFoundError:
        print(f'{lab}: (not found — run the choice-only matrix if you want this baseline)')
print('\n^ compare to the RT runs printed above:')
print('  2a  RT, same 11 tasks  -> direct test: within > 0.15 => RT adds individual signal beyond choice')
print('  2b  RT, all 33 tasks   -> do the RT-ceiling tasks (stroop/simon/ANT/stop-signal) now transfer?')

In [ ]:
# 3) compare to the choice-only baselines
import json
r = '/content/drive/MyDrive/sro_minitaur/results'
labels = {'8b_raw': 'choice-only raw 8B', '': 'choice-only M_pop', 'rt': 'RT model (M_pop_RT)'}
for tag in ['8b_raw', '', 'rt']:
    suf = f'_{tag}' if tag else ''
    try:
        s = json.load(open(f'{r}/surprise_matrix_summary{suf}.json'))
        print(f"{labels[tag]:22s}  within={s['within']:.3f}  across={s['across']:.3f}  chance={s['chance']:.3f}")
    except FileNotFoundError:
        print(f'{labels[tag]}: not found')
print('\nwithin(RT) > within(choice-only ~0.14) => RT carries individual signal: thesis confirmed + a better model.')
print('within(RT) ~= choice-only       => binned-RT tokens did not add transferable signal (try Approach B).')

## Approach B — continuous RT regression head (next, if A is promising)

Instead of discretizing RT into tokens, add a small head that reads M_pop's hidden
state at each decision and predicts **log-RT** (regression, or the params of a
log-normal / shifted-lognormal), trained jointly with the choice loss. More faithful
to continuous RT, but needs model surgery + a custom training loop (doesn't reuse
`train_mpop`). The individual signal would then live in the per-trial RT prediction
residual.

This is a separate build — **ask and I'll add `scripts/train_rt_head.py`** once A shows
whether RT moves the needle. (Approach C = DDM-grounded joint (choice, RT) likelihood —
the most principled, biggest build, closest to the RL/DDM cognitive-modeling line.)